# Short-Term Reversal on MOEX Stocks

Weekly mean reversion strategy: buy the worst-performing stocks over the past week, hold for one week. Long-only, tested on the MOEX top-100 liquid universe from 2022 onward.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

from data_cache import get_stocks, RU_UNIVERSE_100

# Transaction costs (Trader plan)
COMMISSION_PCT = 0.0005
SLIPPAGE_PCT = 0.0008
TOTAL_COST_PCT = (COMMISSION_PCT + SLIPPAGE_PCT) * 2  # round trip ~0.26%

print("Loading daily candles...")
stocks_d = get_stocks(RU_UNIVERSE_100, interval=24, start="2022-01-01", verbose=False)
stocks_d["date"] = stocks_d["timestamp"].dt.date

# Pivot: date x ticker -> close
prices = stocks_d.pivot_table(index="date", columns="ticker", values="close", aggfunc="last")
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()

daily_ret = prices.pct_change()

print(f"Daily bars: {len(prices)}")
print(f"Tickers: {prices.shape[1]}")
print(f"Period: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Avg tickers per day: {prices.notna().sum(axis=1).mean():.0f}")

In [ ]:
# Quintile analysis: sort stocks by past 5d return, measure forward 5d return per quintile.

def rolling_return(prices_df, n_days):
    return prices_df / prices_df.shift(n_days) - 1


def forward_return(prices_df, n_days):
    return prices_df.shift(-n_days) / prices_df - 1


def quintile_forward_analysis(prices_df, signal_days=5, forward_days=5,
                                rebalance_every=5, n_quintiles=5, min_stocks=20):
    sig = rolling_return(prices_df, signal_days)
    fwd = forward_return(prices_df, forward_days)
    
    rebal_dates = prices_df.index[signal_days:-forward_days:rebalance_every]
    
    rows = []
    for d in rebal_dates:
        s = sig.loc[d].dropna()
        f = fwd.loc[d].dropna()
        common = s.index.intersection(f.index)
        if len(common) < min_stocks:
            continue
        
        s = s[common]
        f = f[common]
        try:
            q = pd.qcut(s, n_quintiles, labels=False, duplicates="drop")
        except ValueError:
            continue
        
        row = {"date": d}
        for i in range(n_quintiles):
            mask = (q == i)
            if mask.sum() > 0:
                row[f"Q{i+1}"] = f[mask].mean()
        rows.append(row)
    
    df = pd.DataFrame(rows).set_index("date")
    df["Q1-Q5"] = df["Q1"] - df["Q5"]
    return df


q_basic = quintile_forward_analysis(prices, signal_days=5, forward_days=5, rebalance_every=5)
print(f"Test periods: {len(q_basic)}")
print(f"\nMean forward 5d return by quintile of past 5d return:")
mean_ret = q_basic[["Q1", "Q2", "Q3", "Q4", "Q5"]].mean() * 100
display(mean_ret.round(3))

q1_minus_q5_pct = q_basic["Q1-Q5"].mean() * 100
print(f"\nQ1-Q5 spread: {q1_minus_q5_pct:+.3f}% per 5 days")
print(f"  Q1 = worst past return (buy candidates), Q5 = best past return")
print(f"  Spread > 0 means reversal works")
print(f"  Win rate Q1>Q5: {(q_basic['Q1-Q5'] > 0).mean()*100:.0f}% of periods")
print(f"  t-stat: {q_basic['Q1-Q5'].mean() / q_basic['Q1-Q5'].std() * np.sqrt(len(q_basic)):.2f}")

In [ ]:
# Visualize quintile bar chart, equity curves, and rolling Q1-Q5 spread.

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

q_cols = ["Q1", "Q2", "Q3", "Q4", "Q5"]
mean_ret_pct = q_basic[q_cols].mean() * 100

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, 5))
bars = axes[0].bar(q_cols, mean_ret_pct, color=colors)
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_title(f"Forward 5d return by quintile of past 5d return\n"
                   f"Q1-Q5 spread = {mean_ret_pct['Q1'] - mean_ret_pct['Q5']:.2f}% per 5 days")
axes[0].set_ylabel("Forward 5-day return, %")
for bar, val in zip(bars, mean_ret_pct):
    axes[0].text(bar.get_x() + bar.get_width()/2, val,
                  f"{val:.2f}%", ha="center",
                  va="bottom" if val >= 0 else "top", fontsize=10)
axes[0].set_xlabel("Q1 = worst past losers (buy candidates); Q5 = best past winners")

for i, q in enumerate(q_cols):
    cum = (1 + q_basic[q]).cumprod() * 100 - 100
    axes[1].plot(cum.index, cum, label=q, color=colors[i], linewidth=1.5)

axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_title("Quintile equity curves (before costs)")
axes[1].set_ylabel("Cumulative return, %")
axes[1].legend()

plt.tight_layout()
plt.show()

rolling = q_basic["Q1-Q5"].rolling(12).mean() * 100
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rolling.index, rolling, color="purple", linewidth=1.5)
ax.fill_between(rolling.index, rolling, alpha=0.2, color="purple")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Rolling 12-period Q1-Q5 spread (positive = reversal works)")
ax.set_ylabel("Q1-Q5 spread, %")
plt.tight_layout()
plt.show()

In [ ]:
# Backtest long-only top-N losers/winners across signal/hold/portfolio-size grid.

def backtest_top_n(prices_df, signal_days=5, hold_days=5,
                    top_n=5, side="losers", min_stocks=20,
                    cost_pct=TOTAL_COST_PCT):
    sig = rolling_return(prices_df, signal_days)
    fwd = forward_return(prices_df, hold_days)
    
    rebal_dates = prices_df.index[signal_days:-hold_days:hold_days]
    
    rows = []
    for d in rebal_dates:
        s = sig.loc[d].dropna()
        f = fwd.loc[d].dropna()
        common = s.index.intersection(f.index)
        if len(common) < min_stocks:
            continue
        
        s = s[common]
        f = f[common]
        
        if side == "losers":
            picks = s.nsmallest(top_n).index
        else:
            picks = s.nlargest(top_n).index
        
        port_ret = f[picks].mean() - cost_pct
        rows.append({"date": d, "ret": port_ret})
    
    return pd.DataFrame(rows).set_index("date")["ret"]


def metrics_for_period_returns(ret_series, period_days):
    if len(ret_series) < 5:
        return None
    
    periods_per_year = 252 / period_days
    ann_ret = (1 + ret_series.mean()) ** periods_per_year - 1
    ann_vol = ret_series.std() * np.sqrt(periods_per_year)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    
    cum = (1 + ret_series).cumprod()
    max_dd = ((cum.cummax() - cum) / cum.cummax()).max()
    
    return {
        "n_periods": len(ret_series),
        "ann_ret_pct": ann_ret * 100,
        "ann_vol_pct": ann_vol * 100,
        "sharpe": sharpe,
        "max_dd_pct": max_dd * 100,
        "winrate_pct": (ret_series > 0).mean() * 100,
        "avg_per_period_pct": ret_series.mean() * 100,
    }


results = []
for sig_d in [3, 5, 10]:
    for hold_d in [3, 5, 10]:
        for top_n in [3, 5, 10]:
            for side in ["losers", "winners"]:
                ret = backtest_top_n(prices, signal_days=sig_d, hold_days=hold_d,
                                       top_n=top_n, side=side)
                m = metrics_for_period_returns(ret, hold_d)
                if m and m["n_periods"] >= 10:
                    m["signal_d"] = sig_d
                    m["hold_d"] = hold_d
                    m["top_n"] = top_n
                    m["side"] = side
                    results.append(m)

results_df = pd.DataFrame(results).sort_values("sharpe", ascending=False)
print("Top 15 configurations by Sharpe:")
cols = ["signal_d", "hold_d", "top_n", "side", "n_periods", "ann_ret_pct",
        "sharpe", "max_dd_pct", "winrate_pct"]
display(results_df[cols].head(15).round(3))

print("\nWorst 5:")
display(results_df[cols].tail(5).round(3))

In [ ]:
# Compare losers vs winners vs equal-weight buy-and-hold equity curves across configs.

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

configs_to_compare = [
    (5, 5, 5),
    (3, 3, 5),
    (10, 10, 5),
    (5, 5, 3),
]

for ax, (sig_d, hold_d, top_n) in zip(axes.flat, configs_to_compare):
    losers_ret = backtest_top_n(prices, signal_days=sig_d, hold_days=hold_d,
                                  top_n=top_n, side="losers")
    winners_ret = backtest_top_n(prices, signal_days=sig_d, hold_days=hold_d,
                                   top_n=top_n, side="winners")
    
    eq_rebal = []
    for d in losers_ret.index:
        try:
            idx = prices.index.get_loc(d)
            if idx + hold_d >= len(prices):
                continue
            future_d = prices.index[idx + hold_d]
            day_rets = prices.loc[future_d] / prices.loc[d] - 1
            eq_rebal.append({"date": d, "ret": day_rets.mean() - TOTAL_COST_PCT})
        except KeyError:
            continue
    eq_ret = pd.DataFrame(eq_rebal).set_index("date")["ret"] if eq_rebal else pd.Series(dtype=float)
    
    if not losers_ret.empty:
        cum_l = (1 + losers_ret).cumprod() * 100 - 100
        ax.plot(cum_l.index, cum_l, color="#2ecc71", linewidth=1.8, label="Losers (reversal)")
    if not winners_ret.empty:
        cum_w = (1 + winners_ret).cumprod() * 100 - 100
        ax.plot(cum_w.index, cum_w, color="#e74c3c", linewidth=1.8, label="Winners (momentum)")
    if not eq_ret.empty:
        cum_e = (1 + eq_ret).cumprod() * 100 - 100
        ax.plot(cum_e.index, cum_e, color="grey", linewidth=1.2, alpha=0.7,
                label="Buy&hold equal-weight")
    
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(f"signal={sig_d}d, hold={hold_d}d, top-{top_n}")
    ax.set_ylabel("Cumulative return, %")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Robustness check: split sample in half, compare metrics for best loser config.

losers_only = results_df[results_df["side"] == "losers"]
if not losers_only.empty:
    best = losers_only.iloc[0]
    sig_d, hold_d, top_n = int(best["signal_d"]), int(best["hold_d"]), int(best["top_n"])
    print(f"Best loser config: signal={sig_d}d, hold={hold_d}d, top-{top_n}")
    print(f"  Sharpe: {best['sharpe']:.2f}, Ann ret: {best['ann_ret_pct']:.1f}%")
else:
    sig_d, hold_d, top_n = 5, 5, 5

ret_losers = backtest_top_n(prices, signal_days=sig_d, hold_days=hold_d,
                              top_n=top_n, side="losers")

mid_idx = len(ret_losers) // 2
first_half = ret_losers.iloc[:mid_idx]
second_half = ret_losers.iloc[mid_idx:]

print(f"\n--- Time-split robustness ---")
for label, half in [("First half", first_half), ("Second half", second_half)]:
    m = metrics_for_period_returns(half, hold_d)
    if m:
        print(f"\n{label} ({half.index.min().date()} to {half.index.max().date()}):")
        print(f"  Periods: {m['n_periods']}")
        print(f"  Ann ret: {m['ann_ret_pct']:+.1f}%, Sharpe: {m['sharpe']:.2f}")
        print(f"  Win rate: {m['winrate_pct']:.0f}%")

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

period_pct = ret_losers * 100
colors_p = ["#2ecc71" if x > 0 else "#e74c3c" for x in period_pct]
axes[0].bar(range(len(period_pct)), period_pct, color=colors_p, alpha=0.8)
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].axhline(period_pct.mean(), color="blue", linestyle="--", alpha=0.5,
                label=f"Mean: {period_pct.mean():.2f}%")
axes[0].set_title(f"Per-period PnL (signal={sig_d}d, hold={hold_d}d, top-{top_n}, losers)")
axes[0].set_ylabel("Period return, %")
axes[0].set_xlabel(f"Period # ({hold_d}-day)")
axes[0].legend()

cum = (1 + ret_losers).cumprod()
dd = (cum.cummax() - cum) / cum.cummax() * 100
axes[1].fill_between(dd.index, -dd, 0, color="red", alpha=0.4)
axes[1].set_title(f"Drawdown (max = {dd.max():.1f}%)")
axes[1].set_ylabel("Drawdown, %")

plt.tight_layout()
plt.show()

In [ ]:
# Conditional reversal: only trade when weekly drop exceeds a panic threshold.

def backtest_top_n_filtered(prices_df, signal_days=5, hold_days=5,
                              top_n=5, min_drop_pct=None,
                              min_stocks=20, cost_pct=TOTAL_COST_PCT):
    sig = rolling_return(prices_df, signal_days)
    fwd = forward_return(prices_df, hold_days)
    
    rebal_dates = prices_df.index[signal_days:-hold_days:hold_days]
    
    rows = []
    for d in rebal_dates:
        s = sig.loc[d].dropna()
        f = fwd.loc[d].dropna()
        common = s.index.intersection(f.index)
        if len(common) < min_stocks:
            continue
        s, f = s[common], f[common]
        
        if min_drop_pct is not None:
            candidates = s[s * 100 <= -min_drop_pct]
            if len(candidates) < top_n:
                continue
        else:
            candidates = s
        
        picks = candidates.nsmallest(top_n).index
        port_ret = f[picks].mean() - cost_pct
        rows.append({"date": d, "ret": port_ret})
    
    return pd.DataFrame(rows).set_index("date")["ret"] if rows else pd.Series(dtype=float)


panic_results = []
for thresh in [None, 2.0, 3.0, 5.0, 7.0, 10.0]:
    ret = backtest_top_n_filtered(prices, signal_days=5, hold_days=5,
                                    top_n=5, min_drop_pct=thresh)
    m = metrics_for_period_returns(ret, 5)
    if m and m["n_periods"] >= 5:
        m["min_drop_pct"] = thresh if thresh else 0
        panic_results.append(m)

panic_df = pd.DataFrame(panic_results)
print("Panic filter effect (signal=5d, hold=5d, top-5 losers):")
display(panic_df[["min_drop_pct", "n_periods", "ann_ret_pct", "sharpe",
                    "winrate_pct", "max_dd_pct"]].round(2))

## Results

- **Unconditional weekly reversal is dead on MOEX.** Q1-Q5 spread is negative (-0.16% per week, t=-0.71). Losers underperform winners across all configurations tested.
- **Panic filter rescues the effect.** Restricting to stocks with >=10% weekly drops yields Sharpe 1.09, 84% annualized return, 56% win rate (39 periods). The effect is monotonically increasing with the drop threshold.
- **Practical concern:** the 10% filter fires infrequently (39 out of 240 weeks), limiting capital deployment. Lower thresholds (5-7%) show weak or no edge.
- **Conclusion:** No standalone strategy here. The panic-conditional signal may be useful as a tactical overlay or combined with other strategies.